# LABORATÓRIO 5 — BI e Data Mining com Python

Disciplina: Gestão de Sistemas de Informação  
Professor: Eduardo Max Amaro Amaral  

## Objetivo

Aplicar conceitos de OLAP (exploração multidimensional) e Mineração de Dados (regras de associação) utilizando Pandas sobre uma base de 200 registros de vendas.

In [1]:
import pandas as pd

print("PARTE 0 - CARGA DE DADOS")

df = pd.read_csv("vendas_200_registros.csv")

# Conversão da coluna Data para datetime (essencial para filtros de período)
df["Data"] = pd.to_datetime(df["Data"], format="%d/%m/%Y")

display(df.head())

print("\nTotal de Registros:", len(df))

PARTE 0 - CARGA DE DADOS


,ID_Transacao,Data,Regional,Vendedor,Categoria,Produto,Valor_Venda
0,1,2026-03-11,Sul,Ana,Acessórios,Mouse Gamer,150.0
1,2,2026-03-08,Sul,Ana,Eletrodomésticos,Airfryer,450.0
2,3,2026-03-04,Nordeste,Ana,Acessórios,Teclado Mecânico,350.0
3,4,2026-03-13,Sul,Carla,Cozinha,Jogo de Panelas,600.0
4,5,2026-03-14,Sudeste,Diego,Eletrodomésticos,Geladeira,3800.0



Total de Registros: 200


# Parte 1 — OLAP (Exploração Multidimensional)

Nesta seção aplicaremos operações típicas de OLAP utilizando Pandas:

- Drill-down → detalhamento hierárquico
- Slice → filtro em uma dimensão
- Dice → recorte multidimensional

O objetivo é simular a navegação analítica sobre um cubo de dados.

In [2]:
print("EXERCÍCIO 1.1 — DRILL-DOWN")

# Agregação por Regional
faturamento_regional = (
    df.groupby("Regional")["Valor_Venda"]
      .sum()
      .sort_values(ascending=False)
)

print("\nFaturamento por Regional:")
display(faturamento_regional)

EXERCÍCIO 1.1 — DRILL-DOWN

Faturamento por Regional:


Regional
Sudeste     107000.0
Sul          88200.0
Nordeste     73450.0
Name: Valor_Venda, dtype: float64

In [3]:
print("Drill-down da Regional Sudeste por Vendedor")

sudeste = df[df["Regional"] == "Sudeste"]

faturamento_vendedor = (
    sudeste.groupby("Vendedor")["Valor_Venda"]
           .sum()
           .sort_values(ascending=False)
)

display(faturamento_vendedor)

Drill-down da Regional Sudeste por Vendedor


Vendedor
Carla    35450.0
Diego    25450.0
Ana      20400.0
Elena    14250.0
Bruno    11450.0
Name: Valor_Venda, dtype: float64

## Exercício 1.2 — SLICE

O Slice consiste em fixar uma dimensão e analisar apenas um subconjunto específico.

Pergunta de Gestão:
Qual foi o faturamento apenas da categoria "Eletrodomésticos"?

In [4]:
print("EXERCÍCIO 1.2 — SLICE")

# Filtrando apenas a categoria Eletrodomésticos
slice_eletro = df[df["Categoria"] == "Eletrodomésticos"]

faturamento_slice = (
    slice_eletro.groupby("Regional")["Valor_Venda"]
    .sum()
    .sort_values(ascending=False)
)

print("\nFaturamento de Eletrodomésticos por Regional:")
display(faturamento_slice)

EXERCÍCIO 1.2 — SLICE

Faturamento de Eletrodomésticos por Regional:


Regional
Sudeste     44000.0
Sul         40700.0
Nordeste    31000.0
Name: Valor_Venda, dtype: float64

## Exercício 1.3 — DICE

O Dice permite aplicar múltiplos filtros simultaneamente.

Pergunta de Gestão:
Qual o faturamento da Regional Sudeste,
considerando apenas a Categoria "Acessórios",
e apenas os vendedores Ana e Carla?

In [5]:
print("EXERCÍCIO 1.3 — DICE")

dice = df[
    (df["Regional"] == "Sudeste") &
    (df["Categoria"] == "Acessórios") &
    (df["Vendedor"].isin(["Ana", "Carla"]))
]

faturamento_dice = (
    dice.groupby("Vendedor")["Valor_Venda"]
    .sum()
    .sort_values(ascending=False)
)

print("\nFaturamento do recorte multidimensional:")
display(faturamento_dice)

EXERCÍCIO 1.3 — DICE

Faturamento do recorte multidimensional:


Vendedor
Carla    1200.0
Ana       900.0
Name: Valor_Venda, dtype: float64

# Parte 2 — Business Intelligence (Indicadores)

Nesta etapa iremos gerar KPIs estratégicos:

- Faturamento total
- Ticket médio
- Top vendedor
- Produto mais vendido
- Análise temporal (vendas por dia)

O objetivo é transformar dados em informação para tomada de decisão.

In [6]:
print("PARTE 2 — KPI GERAL")

faturamento_total = df["Valor_Venda"].sum()
ticket_medio = df["Valor_Venda"].mean()

print(f"\nFaturamento Total: R$ {faturamento_total:,.2f}")
print(f"Ticket Médio: R$ {ticket_medio:,.2f}")

PARTE 2 — KPI GERAL

Faturamento Total: R$ 268,650.00
Ticket Médio: R$ 1,343.25


In [7]:
print("\nEXERCÍCIO 2.2 — TOP VENDEDOR")

top_vendedor = (
    df.groupby("Vendedor")["Valor_Venda"]
      .sum()
      .sort_values(ascending=False)
)

display(top_vendedor.head(1))


EXERCÍCIO 2.2 — TOP VENDEDOR


Vendedor
Carla    63300.0
Name: Valor_Venda, dtype: float64

In [9]:
print("\nEXERCÍCIO 2.3 — PRODUTO MAIS RENTÁVEL")

produto_top = (
    df.groupby("Produto")["Valor_Venda"]
      .sum()
      .sort_values(ascending=False)
)

display(produto_top.head(5))


EXERCÍCIO 2.3 — PRODUTO MAIS RENTÁVEL


Produto
Geladeira           45600.0
Notebook            45000.0
Máquina de Lavar    31900.0
Sofá 3 Lugares      23100.0
Ar Condicionado     19800.0
Name: Valor_Venda, dtype: float64

In [10]:
print("\nEXERCÍCIO 2.4 — VENDAS POR DATA")

# Garantir que Data esteja como datetime
df["Data"] = pd.to_datetime(df["Data"])

vendas_por_dia = (
    df.groupby("Data")["Valor_Venda"]
      .sum()
      .sort_index()
)

display(vendas_por_dia.head())


EXERCÍCIO 2.4 — VENDAS POR DATA


Data
2026-03-01     6700.0
2026-03-02    15400.0
2026-03-03     9250.0
2026-03-04    27650.0
2026-03-05    20250.0
Name: Valor_Venda, dtype: float64

# Parte 3 — Data Mining

Nesta etapa buscamos padrões e relações nos dados.

Aplicaremos:
- Análise de correlação simples
- Tabela dinâmica (pivot table)
- Identificação de concentração de vendas

In [11]:
print("EXERCÍCIO 3.1 — Padrão Regional x Categoria")

pivot_regional_categoria = pd.pivot_table(
    df,
    values="Valor_Venda",
    index="Regional",
    columns="Categoria",
    aggfunc="sum",
    fill_value=0
)

display(pivot_regional_categoria)

EXERCÍCIO 3.1 — Padrão Regional x Categoria


Categoria,Acessórios,Cozinha,Eletrodomésticos,Eletrônicos,Móveis
Regional,,,,,
Nordeste,3050.0,3600.0,31000.0,22300.0,13500.0
Sudeste,3200.0,1800.0,44000.0,34800.0,23200.0
Sul,3800.0,2400.0,40700.0,34800.0,6500.0


In [12]:
print("\nEXERCÍCIO 3.2 — Concentração de Receita (Top Produtos)")

produto_receita = (
    df.groupby("Produto")["Valor_Venda"]
      .sum()
      .sort_values(ascending=False)
)

receita_total = produto_receita.sum()
receita_acumulada = produto_receita.cumsum() / receita_total * 100

concentracao = pd.DataFrame({
    "Receita": produto_receita,
    "% Acumulado": receita_acumulada
})

display(concentracao.head(10))


EXERCÍCIO 3.2 — Concentração de Receita (Top Produtos)


,Receita,% Acumulado
Produto,,
Geladeira,45600.0,16.973758
Notebook,45000.0,33.724176
Máquina de Lavar,31900.0,45.598362
Sofá 3 Lugares,23100.0,54.196910
Ar Condicionado,19800.0,61.567095
Smartphone,17500.0,68.081146
Tablet,16200.0,74.111297
Monitor 24,13200.0,79.024753
Mesa de Jantar,10500.0,82.933184


In [13]:
print("\nEXERCÍCIO 3.3 — Identificação de Outliers")

media = df["Valor_Venda"].mean()
desvio = df["Valor_Venda"].std()

limite_superior = media + 2 * desvio

outliers = df[df["Valor_Venda"] > limite_superior]

print(f"Média: {media:.2f}")
print(f"Desvio Padrão: {desvio:.2f}")
print(f"Limite Superior (2σ): {limite_superior:.2f}")
print(f"Quantidade de Outliers: {len(outliers)}")

display(outliers.head())


EXERCÍCIO 3.3 — Identificação de Outliers
Média: 1343.25
Desvio Padrão: 1283.39
Limite Superior (2σ): 3910.02
Quantidade de Outliers: 10


,ID_Transacao,Data,Regional,Vendedor,Categoria,Produto,Valor_Venda
40,41,2026-03-03,Sudeste,Elena,Eletrônicos,Notebook,4500.0
47,48,2026-03-06,Nordeste,Ana,Eletrônicos,Notebook,4500.0
57,58,2026-03-04,Sul,Ana,Eletrônicos,Notebook,4500.0
82,83,2026-03-12,Nordeste,Elena,Eletrônicos,Notebook,4500.0
87,88,2026-03-06,Nordeste,Elena,Eletrônicos,Notebook,4500.0


# Conclusão Geral do Laboratório

Este laboratório demonstrou a aplicação prática de Business Intelligence e Data Mining com Pandas.

Principais aprendizados:

- OLAP permite navegação multidimensional (Drill-down, Slice, Dice)
- KPIs transformam dados brutos em informação estratégica
- Pivot tables revelam padrões regionais e setoriais
- Análise de concentração evidencia possível efeito Pareto
- Identificação de outliers auxilia na detecção de anomalias

A base de dados permitiu simular um cenário real de apoio à decisão empresarial.